[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drksci/phosphene/blob/main/notebooks/03_train_vtm.ipynb)

# 03 · Train the VTM

Per-cell segmentation transformer (conv stem + axial row/column attention, ~2M params) trained on
`synth_train` (exact) + `weak_*` (heuristic, confidence-weighted ×0.5) + `llm_train` (×2 oversampled).
Evaluated on the held-out reviewer-labeled **gold set** against the heuristic baseline, then exported
to ONNX for in-browser streaming (onnxruntime-web / WebGPU). Use a **T4 GPU** runtime.

In [ ]:
#@title Setup: clone + install (re-run safe)
import os, sys
if not os.path.exists("/content/phosphene"):
    !git clone -q https://github.com/drksci/phosphene /content/phosphene
%cd /content/phosphene
!git pull -q
!pip install -q -e packages/vtcore -e packages/vttui -e "packages/vtm[train,data,label]"

#@markdown Persist data/models to Google Drive (survives runtime resets) or keep them in /content.
USE_DRIVE = False  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/phosphene"
else:
    DATA = "/content/phosphene_data"
os.makedirs(DATA, exist_ok=True)
print("DATA =", DATA)

In [ ]:
#@title Parameters
EPOCHS = 8      #@param {type:"integer"}
BATCH = 16      #@param {type:"integer"}
D_MODEL = 128   #@param {type:"integer"}
LAYERS = 4      #@param {type:"integer"}
USE_WEAK = True #@param {type:"boolean"}

In [ ]:
import glob, json
have = lambda n: os.path.exists(f"{DATA}/{n}.npz")
train_shards = [f"{DATA}/synth_train.npz"]
if USE_WEAK: train_shards += sorted(glob.glob(f"{DATA}/weak_*.npz"))
if have("llm_train"): train_shards += [f"{DATA}/llm_train.npz"] * 2
val_shards = [f"{DATA}/synth_val.npz"] + ([f"{DATA}/llm_val.npz"] if have("llm_val") else [])
print(train_shards, val_shards)

In [ ]:
from vtm.train import train
model = train(train_shards, val_shards, out_dir=f"{DATA}/runs/vtm", epochs=EPOCHS, batch_size=BATCH, d=D_MODEL, layers=LAYERS)

In [ ]:
#@title Evaluate: model vs heuristics (gold = reviewer labels; synth = exact)
from vtcore import load_frames
from vtm.train import predict
from vtm.labeling.heuristics import label
from vtm.metrics import confusion, report
results = {}
for split in ["gold_test", "synth_val"]:
    if not have(split): continue
    fr, ex = load_frames(f"{DATA}/{split}.npz")
    pm = [predict(model, f)[0] for f in fr]
    ph = [label(f)[0] for f in fr]
    results[split] = {"model": report(confusion(pm, ex["roles"])), "heuristics": report(confusion(ph, ex["roles"]))}
    print(split, {k: round(v["macro_iou"], 3) for k, v in results[split].items()})
json.dump(results, open(f"{DATA}/runs/vtm/eval.json", "w"), indent=1)

In [ ]:
#@title Active learning: model confidence on the unlabeled pool → feed back into 02
import numpy as np
from vtcore import load_frames
conf = []
for p in sorted(glob.glob(f"{DATA}/weak_*.npz")):
    for f in load_frames(p)[0]:
        r, c, _ = predict(model, f)
        m = r != 0
        conf.append(float(c[m].mean()) if m.any() else 1.0)
json.dump(conf, open(f"{DATA}/model_conf.json", "w"))
print("pool mean confidence", np.mean(conf).round(3), "→ re-run 02 to label the least-confident frames")

In [ ]:
#@title Export ONNX (browser / edge inference)
from vtm.train import export_onnx
export_onnx(model, f"{DATA}/runs/vtm/vtm.onnx")
!ls -la {DATA}/runs/vtm